# Cosmos DB Advance Search 
##  with Azure OpenAI 

This notebook walks through an end‑to‑end example of using **Azure Cosmos DB for NoSQL** together with **Azure OpenAI**:

- Set up authentication and clients
- Create Cosmos DB containers (simple, products, and PDF chunks)
- Generate embeddings with Azure OpenAI
- Store and query vector data (pure vector, full‑text, and hybrid search)
- Run a mini CRUD flow on a sample product
- Ingest PDF chunks with embeddings and query them via vector search.

### Install required SDK packages

Installs the `azure-cosmos` SDK for Cosmos DB and the `openai` package for calling Azure OpenAI.

In [1]:
%pip install azure-cosmos>=4.15.0
%pip install openai
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Environment setup and configuration

The next block:

- Installs required Python packages (`azure-cosmos` for Cosmos DB, `openai` for Azure OpenAI)
- Imports common libraries (asyncio, requests, logging, typing helpers)
- Imports Azure Cosmos DB async SDK types and Azure OpenAI client
- Defines key configuration values: Cosmos DB endpoint & database, Azure OpenAI endpoint, model names, embedding dimensions, and API key.

Together, these cells prepare the runtime so that all later examples can connect to Cosmos DB and Azure OpenAI.

### Import libraries 

Imports standard Python libraries, Cosmos DB and Azure OpenAI SDKs

In [2]:
import asyncio
import json
import requests
import logging
import base64
from typing import Optional, List, Dict, Any
import uuid
import random

import os
from dotenv import load_dotenv
from pathlib import Path

from rich.pretty import pprint

from azure.cosmos.aio import CosmosClient
from azure.cosmos import PartitionKey, ThroughputProperties, exceptions

from openai.lib.azure import AsyncAzureOpenAI

from azure.core.credentials import AccessToken, TokenCredential
base64

<module 'base64' from '/home/trusted-service-user/jupyter-env/python3.12/lib/python3.12/base64.py'>

### Define configuration
sets key configuration values such as endpoints, model names, and API keys.

In [3]:
load_dotenv(f"{notebookutils.nbResPath}/builtin/.env.txt")

CosmosDB_Endpoint =  os.getenv("COSMOSDB_ENDPOINT")
CosmosDB_Database = os.getenv("COSMOSDB_DATABASE")
CosmosDB_CredentialScope = os.getenv("COSMOSDB_CREDENTIAL_SCOPE")

OpenAI_Endpoint = os.getenv("OPENAI_ENDPOINT")
OpenAI_CompletionModelName = os.getenv("OPENAI_COMPLETION_MODEL_NAME", "gpt-5-chat")
OpenAI_APIVersion = os.getenv("OPENAI_API_VERSION")
OpenAI_EmbeddingModelName = os.getenv("OPENAI_EMBEDDING_MODEL_NAME")
OpenAI_EmbeddingDimensions = int(os.getenv("OPENAI_EMBEDDING_DIMENSIONS", "1536"))
OpenAI_Key = os.getenv("OPENAI_KEY")

### Authentication helper

This section defines `FabricTokenCredential`, a small wrapper around Fabric’s `notebookutils.credentials.getToken` so that it can be used as an `azure.core.credentials.TokenCredential` with Azure SDK clients (like Cosmos DB). It:

- Stores the target scope (e.g., Cosmos DB)
- Fetches an access token for that scope
- Returns it as an `AccessToken` object so Cosmos DB can authenticate using your Fabric identity.

In [4]:
class FabricTokenCredential(TokenCredential):
    """Token credential for Fabric Cosmos DB access with automatic refresh and retry logic."""
    
    def get_token(self, *scopes: str, claims: Optional[str] = None, tenant_id: Optional[str] = None,
                  enable_cae: bool = False, **kwargs: Any) -> AccessToken:
        access_token = notebookutils.credentials.getToken("https://cosmos.azure.com/.default")
        parts = access_token.split(".")
        if len(parts) < 2:
            raise ValueError("Invalid JWT format")
        payload_b64 = parts[1]
        # Fix padding
        padding = (-len(payload_b64)) % 4
        if padding:
            payload_b64 += "=" * padding
        payload_json = base64.urlsafe_b64decode(payload_b64.encode("utf-8")).decode("utf-8")
        payload = json.loads(payload_json)
        exp = payload.get("exp")
        if exp is None:
            raise ValueError("exp claim missing in token")
        return AccessToken(token=access_token, expires_on=exp)

## Create Azure Cosmos DB client and database client

### Create Cosmos DB client

Initializes the async `CosmosClient` using Fabric credentials and gets a reference to the target database.

In [5]:
print(f"connecting to {CosmosDB_Endpoint}")
CosmosDBClient = CosmosClient(CosmosDB_Endpoint, FabricTokenCredential(CosmosDB_CredentialScope))
CosmosDBDatabase = CosmosDBClient.get_database_client(CosmosDB_Database)

connecting to https://2a5e7999-5340-4eb0-b584-dc225df0876e.z2a.msit-sql.cosmos.fabric.microsoft.com:443/


In [6]:
print(CosmosDB_Endpoint)
print(CosmosDB_Database)

https://2a5e7999-5340-4eb0-b584-dc225df0876e.z2a.msit-sql.cosmos.fabric.microsoft.com:443/
agentic_cosmos_db


## Create Azure OpenAI client

### Create Azure OpenAI client

Builds an `AsyncAzureOpenAI` client bound to your Azure OpenAI endpoint, API key, and API version.

In [7]:
OpenAIClient = AsyncAzureOpenAI(api_version=OpenAI_APIVersion, azure_endpoint=OpenAI_Endpoint, api_key=OpenAI_Key)

## Sample container for basic CRUD

This section defines and runs `create_SampleContainer()`, which:

- Creates a simple Cosmos DB container named `SimpleContainer` (if it does not already exist)
- Uses `/id` as the partition key and configures autoscale throughput
- Prints information about the container and its current throughput

It’s intended as a minimal example for basic create/read/delete and query operations.

In [8]:
async def create_SampleContainer():

    CosmosDB_ContainerName = "SimpleContainer"
    CosmosDB_PartitionKeyPath = "/customerId"
    CosmosDB_AutoscaleMax = 1000

    credential = FabricTokenCredential(CosmosDB_CredentialScope)

    autoscale_props = ThroughputProperties(
        auto_scale_max_throughput=CosmosDB_AutoscaleMax
    )

    container = await CosmosDBDatabase.create_container_if_not_exists(
        id=CosmosDB_ContainerName,
        partition_key=PartitionKey(path=CosmosDB_PartitionKeyPath),
        offer_throughput=autoscale_props
    )

    print(f"Container ready: {container.id}")
    print(f"Partition key: {CosmosDB_PartitionKeyPath}")

    throughput = await container.get_throughput()
    print("Throughput:", throughput)

    print(f"Container Created")
    return container


### Run sample container creation

Executes `create_SampleContainer()` to ensure the simple demo container exists before doing CRUD operations.

In [9]:
sampleContainer = await create_SampleContainer()

Container ready: SimpleContainer
Partition key: /customerId
Throughput: <azure.cosmos.offer.ThroughputProperties object at 0x7e99d3c4bb30>
Container Created


## Create a random sampleCustomer Order document 

In [10]:
# Random customer and order details

async def generate_randomCustomerOrder():

    orderId = str(uuid.uuid4())
    customerId = f"cust-{orderId[-6:]}"
    items1purchasePrice = round(random.uniform(1, 999), 2)
    items1quantity= round(random.uniform(1, 10), 0)
    items2purchasePrice = round(random.uniform(1, 999), 0)
    items2quantity= round(random.uniform(1, 10), 0)
    totalAmount = items1purchasePrice * items1quantity + items2purchasePrice * items2quantity

    customer_order = {
        "id": orderId,
        "customerId": customerId, 
        "orderDate": "2025-10-15T10:30:00",
        "docType": "customerOrder",
        "status": "shipped",
        "totalAmount": totalAmount,
        "shippingAddress": {
            "street": "123 Main St",
            "city": "Seattle", 
            "state": "WA",
            "zipCode": "98101"
        },
        "items": [
            {
                "productId": "a74e7af9-7e13-40cd-90df-7b5e172a8acc",
                "productName": "HyperType Pro K100 RGB Mechanical Keyboard",
                "quantity": items1quantity,
                "purchasePrice": items2purchasePrice,
                "categoryName": "Peripherals, Keyboards"
            },
            {
                "productId": "f9619b13-30a7-4ad4-ae00-9817576fba81",
                "productName": "InfinityCore Apex 12 Pro 5G",
                "quantity": items2quantity, 
                "purchasePrice": items2purchasePrice,
                "categoryName": "Devices, Smartphones"
            }
        ],
        "paymentMethod": "credit_card",
        "trackingNumber": "1Z999AA1234567890"
    }

    return customer_order


### Create Cosmsos DB item based on sample document

In [11]:
customerOrder = await generate_randomCustomerOrder()
print(f" Order ID: {customerOrder['id']}")
print(f"   For customer: {customerOrder['customerId']}")
print(f"   Total: ${customerOrder['totalAmount']}")

customerOrder = await sampleContainer.create_item(customerOrder)

RUCharge = customerOrder.get_response_headers().get("x-ms-request-charge")

print(f"  Request charge: {RUCharge} RUs")
print(f"  Customer order created successfully!")

orderid = customerOrder['id']
customerid= customerOrder['customerId']

 Order ID: 12a5ead9-73ff-4e1e-954e-f38b8df62afc
   For customer: cust-f62afc
   Total: $6539.55
  Request charge: 13.71 RUs
  Customer order created successfully!


### Read Cosmsos DB item based on id and partition key (point read)

In [12]:
#Point read example - Get the customer order by ID and partition key (customerId)

# Point read using read_item method
customerOrder= await sampleContainer.read_item(item=orderid, partition_key=customerid)
RUCharge = customerOrder.get_response_headers().get("x-ms-request-charge")

# Display the results
print(f"Order ID: {customerOrder['id']}")
print(f"  Customer: {customerOrder['customerId']}")
print(f"  Total: ${customerOrder['totalAmount']}")

print(f"  Request charge: {RUCharge} RUs")

orderid = customerOrder['id']
customerid= customerOrder['customerId']


Order ID: 12a5ead9-73ff-4e1e-954e-f38b8df62afc
  Customer: cust-f62afc
  Total: $6539.55
  Request charge: 1 RUs


### Read Cosmsos DB the same item based on id and partition key with a SELECT (query operation)

In [13]:
RUCharges = []
def CaptureRU(headers, response):
    RUCharges.append(float(headers.get("x-ms-request-charge", 0)))

Query = """
SELECT * FROM c
"""

selectedItems = sampleContainer.query_items(
    query=Query,
    response_hook=CaptureRU
)

async for customerOrder in selectedItems:
    print(f"Order ID: {customerOrder['id']}")
    print(f"Customer: {customerOrder['customerId']}")
    print(f"Total: ${customerOrder['totalAmount']}")
    print()
    orderid = customerOrder['id']
    customerid= customerOrder['customerId']

print(f"  Request charge: {sum(RUCharges)} RUs")



Order ID: 12a5ead9-73ff-4e1e-954e-f38b8df62afc
Customer: cust-f62afc
Total: $6539.55

  Request charge: 2.27 RUs


### Query Cosmsos DB for all items

In [16]:
RUCharges = []
def CaptureRU(headers, response):
    RUCharges.append(float(headers.get("x-ms-request-charge", 0)))

Query = """
SELECT * FROM c
WHERE c.id = @orderId AND c.customerId = @customerId
"""

parameters = [
    {"name": "@orderId", "value": orderid},
    {"name": "@customerId", "value": customerid}
]

selectedItems = sampleContainer.query_items(
    query=Query,
    parameters=parameters,
    response_hook=CaptureRU
)

customerOrder = None
async for item in selectedItems:
    customerOrder = item
    break # Since this should return a single item, get the first result

if customerOrder:
    print(f"Order ID: {customerOrder['id']}")
    print(f"Customer: {customerOrder['customerId']}")
    print(f"Total: ${customerOrder['totalAmount']}")
else:
    print("Order not found")

print(f"  Request charge: {sum(RUCharges)} RUs")


Order ID: 12a5ead9-73ff-4e1e-954e-f38b8df62afc
Customer: cust-f62afc
Total: $6539.55
  Request charge: 2.93 RUs


In [17]:
print(f"deleting order {orderid} for customer {customerid}")

try:
    await sampleContainer.delete_item(item=orderid, partition_key=customerid)
    print(f" deleted")

except exceptions.CosmosResourceNotFoundError:
    print(f"item not found")


deleting order 12a5ead9-73ff-4e1e-954e-f38b8df62afc for customer cust-f62afc
 deleted


## Exploring Embeddings and Cosmos DB Vector Search

### Define helper to generate text embeddings

Adds `generate_embedding(text)`, a small async helper that calls Azure OpenAI to turn input text into a numeric embedding vector.

In [18]:
async def generate_embedding(text):

    response = await OpenAIClient.embeddings.create(
        input=text,
        model=OpenAI_EmbeddingModelName
    )
    
    embeddings = response.model_dump()
    return embeddings['data'][0]['embedding']

### Quick embedding sanity check

Calls `generate_embedding("This is a test")` and prints a small slice of the embedding to verify the model call works.

In [19]:
embeddingVector = await generate_embedding("This is a test")
print(f"embedding : {embeddingVector[:2]} ... { embeddingVector[-2:]} ")

embedding : [-0.008058104664087296, -0.0036456480156630278] ... [0.014688358642160892, 0.00228059571236372] 


### ** Todo: change to diskANN

In [20]:
async def create_ProductsContainer():

    CosmosDB_ContainerName = "ProductsContainer"
    CosmosDB_PartitionKeyPath = "/categoryName"
    CosmosDB_AutoscaleMax = 1000
    
    # Define the vector policy for the container
    vector_embedding_policy = {
        "vectorEmbeddings": [
            {
                "path":"/vectors",
                "dataType":"float32",
                "distanceFunction":"cosine",
                "dimensions":1536
            }
        ]
    }
    # Define the full text policy for the container
    full_text_policy = {
   "defaultLanguage": "en-US",
   "fullTextPaths": [
       {
           "path": "/description",
           "language": "en-US"
       },
       {
           "path": "/name",
           "language": "en-US"
       }
   ]
}

    # Define the indexing policy for the container
    indexing_policy = {
        "includedPaths": [
            {
                "path": "/*"
            }
        ],
        "excludedPaths": [
            {
                "path": "/vectors/*"
            },
            {
                "path": "/\"_etag\"/?"
            }
        ],

        "vectorIndexes": [
            {
                "path": "/vectors",
                "type": "quantizedFlat"
            }
        ],

        "fullTextIndexes": [
            {
                "path": "/description",
            },
            {
                "path": "/name",
            }
        ]
    }

    # Create the vectorized sample product container
    container = await CosmosDBDatabase.create_container_if_not_exists(
        id=CosmosDB_ContainerName,
        partition_key=PartitionKey(path=CosmosDB_PartitionKeyPath, kind='Hash'),
        indexing_policy=indexing_policy,
        full_text_policy=full_text_policy,
        vector_embedding_policy=vector_embedding_policy,
        offer_throughput=ThroughputProperties(auto_scale_max_throughput=5000))

    print(f"Container Created")
    return container

### Create or load products container

Runs `create_ProductsContainer()` and stores the resulting container client in `ProductsContainer`.

In [21]:
# Run the function and get the container reference
ProductsContainer = await create_ProductsContainer()

Container Created


In [22]:
async def load_ProductsContainer():
    print("Loading products")
  
    # Load the vectorized product data. Vectors generated using text-3-large model with 512 dimensions
    url = "https://raw.githubusercontent.com/AzureCosmosDB/cosmos-fabric-samples/refs/heads/main/datasets/fabricSampleDataVectors-3-large-512.json"
    data = requests.get(url).json()
    
    # Insert the data into the container
    itemCount = 0
    itemsSkipped = 0
    itemEmbeddings = 0
    
    for item in data:
        #print(f"Processing item: {item.get('id', 'unknown')}")
        try:
            docType = item.get("docType")

            if docType == "product":
                name =item.get("name")
                description =item.get("description")
                if name and description:
                    embeddingText = f"{name} {description}"
                else:
                    embeddingText = f"" 
                embedding = await generate_embedding(embeddingText)
                item["vectors"] = embedding
                itemEmbeddings += 1
            
            await ProductsContainer.create_item(item)
            itemCount += 1
        
        except exceptions.ResourceExistsError:
            itemsSkipped += 1


    print(f"Products loaded {itemCount} items with {itemEmbeddings} embeddings generated and {itemsSkipped} items skipped.")



### Load products data with embeddings

Invokes `load_ProductsContainer()` to pull sample products from GitHub, generate embeddings, and insert them into `ProductsContainer`.

In [23]:
# Run the function ato load the products data
await load_ProductsContainer()

Loading products
Products loaded 832 items with 180 embeddings generated and 0 items skipped.


In [24]:
#Define function to perform vector search
async def vectorsearch_products(search_text: str, similarity: float = 0.7, limit: int = 5) -> List[Dict[str, Any]]:

    try:
        # Generate embeddings for the search text
        embeddings = await generate_embedding(search_text.strip())

        # Cosmos query with VectorDistance to perform similarity search
        query = """
            SELECT TOP @limit 
                VectorDistance(c.vectors, @embeddings) AS SimilarityScore,
                c.name, 
                c.description,
                c.categoryName,
                c.currentPrice,
                c.inventory,
                c.priceHistory
            FROM c 
            WHERE 
                c.docType = @docType
                 AND
                VectorDistance(c.vectors, @embeddings) >= @similarity
            ORDER BY 
                VectorDistance(c.vectors, @embeddings)
        """

        parameters = [
            {"name": "@limit", "value": limit},
            {"name": "@embeddings", "value": embeddings},
            {"name": "@docType", "value": "product"},
            {"name": "@similarity", "value": similarity}
        ]

        # Async query: gather results into a list
        products = [p async for p in ProductsContainer.query_items(
            query=query,
            parameters=parameters
        )]

        # Remove the vectors property if it appears, unnecessarily large
        for p in products:
            p.pop('vectors', None)

        return products
        
    except exceptions.CosmosHttpResponseError as e:
        logging.error(f"Cosmos DB query failed: {e}")
        raise

    except Exception as e:
        logging.error(f"Unexpected error in search_products: {e}")
        raise



### Example: pure vector search over products

Uses `vectorsearch_products` to find products similar to the phrase "gaming desktop" and displays the results.

In [25]:
products = await vectorsearch_products(search_text="gaming desktop",similarity= 0.6, limit=5)
print(f"Found {len(products)} products matching the search criteria.")
display(products) # for tabular output
# pprint(products) # for json friendly output

Found 5 products matching the search criteria.


### Full‑text filter search helper

This block defines `fulltextfilter_products`, which:

- Accepts a free‑text search string and a result limit
- Splits the search text into individual words and builds a `FullTextContainsAny` filter
- Queries the `ProductsContainer` for items with `docType = 'product'` whose `description` matches any of the search terms
- Returns a list of products without the large `vectors` field to keep results compact.

The following cell shows how to call this helper and display the matching products.

In [26]:
#Define function to perform full text search
async def fulltextfilter_products(search_text: str, limit: int = 5) -> List[Dict[str, Any]]:

    try:
        fullTextQuery = f"'{"', '".join(search_text.strip().split())}'"
        query = f"""
            SELECT TOP @limit  c.name, 
                    c.description,
                    c.categoryName,
                    c.currentPrice,
                    c.inventory,
                    c.priceHistory
            FROM c
            WHERE 
                c.docType = @docType 
               AND
                FullTextContainsAny(c.description,{fullTextQuery})
        """
        parameters = [
            {"name": "@limit", "value": limit},
            {"name": "@docType", "value": "product"}
        ]

        #print(f"Full text search for: {search_text}")
        #print(f"Full text query : {fullTextQuery}")
        #print(f"Query : {query}")
        
        # Async query: gather results into a list
        products = [p async for p in ProductsContainer.query_items(
            query=query,
            parameters=parameters
        )]

        # Remove the vectors property if it appears, unnecessarily large
        for p in products: p.pop('vectors', None)

        return products
        
    except exceptions.CosmosHttpResponseError as e:
        logging.error(f"Cosmos DB query failed: {e}")
        raise

    except Exception as e:
        logging.error(f"Unexpected error in search_products: {e}")
        raise



### Example: filtered full‑text search

Runs `fulltextfilter_products` for a specific query string and shows the matched products.

In [27]:
products = await fulltextfilter_products(search_text="gaming pc with GX550", limit=5)
print(f"Found {len(products)} products matching the search criteria.")
display(products) # for tabular output
# pprint(products) # for json friendly output   

Found 5 products matching the search criteria.


## Check status of index transformation 

### Check indexing transformation progress

Reads the products container with quota information and prints how far index transformation has progressed.

In [28]:

# Read container with quota info enabled
response = await ProductsContainer.read(populate_quota_info=True)

# Extract index transformation progress header
headers = ProductsContainer.client_connection.last_response_headers
progress = headers.get(
    "x-ms-documentdb-collection-index-transformation-progress"
)

print(f"Index transformation progress: {progress} %")


Index transformation progress: 100 %


### Ranked full‑text search helper

This block defines `fulltextsearch_products`, which:

- Builds the same kind of full‑text term list as `fulltextfilter_products`
- Selects products with `docType = 'product'`
- Orders results using `RANK FullTextScore(...)` so the most relevant matches appear first
- Returns concise product documents (without embeddings).

The next cell demonstrates using this ranked full‑text search for a sample query.

In [29]:
#Define function to perform full text search
async def fulltextsearch_products(search_text: str, limit: int = 5) -> List[Dict[str, Any]]:

    try:
        fullTextQuery = f"'{"', '".join(search_text.strip().split())}'"
        query = f"""
            SELECT TOP @limit  
                    c.name, 
                    c.description,
                    c.categoryName,
                    c.currentPrice,
                    c.inventory,
                    c.priceHistory
            FROM c
            WHERE 
                c.docType = @docType 
            ORDER BY 
                RANK FullTextScore(c.description,{fullTextQuery})
        """
        parameters = [
            {"name": "@limit", "value": limit},
            {"name": "@docType", "value": "product"}
        ]

        #print(f"Full text search for: {search_text}")
        #print(f"Full text query : {fullTextQuery}")
        #print(f"Query : {query}")
        
        # Async query: gather results into a list
        products = [p async for p in ProductsContainer.query_items(
            query=query,
            parameters=parameters
        )]

        # Remove the vectors property if it appears, unnecessarily large
        for p in products: p.pop('vectors', None)

        return products
        
    except exceptions.CosmosHttpResponseError as e:
        logging.error(f"Cosmos DB query failed: {e}")
        raise

    except Exception as e:
        logging.error(f"Unexpected error in search_products: {e}")
        raise


### Example: ranked full‑text search

Calls `fulltextsearch_products` with a sample query to retrieve products ordered by full‑text relevance.

In [30]:
products = await fulltextsearch_products(search_text="gaming pc with GX550", limit=5)
print(f"Found {len(products)} products matching the search criteria.")
display(products) # for tabular output
# pprint(products) # for json friendly output

Found 5 products matching the search criteria.


### Hybrid vector + full‑text search helper

This section defines `hybridsearch_products`, which:

- Generates an embedding for the user’s query
- Builds a full‑text query string from the search text
- Filters products by minimum vector similarity
- Uses `RRF` to combine vector similarity (`VectorDistance`) and full‑text relevance (`FullTextScore`) into a single hybrid ranking
- Returns a list of top‑ranked products with key fields only.

The following cell shows how to run a hybrid search and inspect the results.

In [31]:
#Define function to perform vector search
async def hybridsearch_products(search_text: str, similarity: float = 0.5, limit: int = 5) -> List[Dict[str, Any]]:

    try:
         # Generate embeddings for the search text
        embeddings = await generate_embedding(search_text.strip())


        fullTextQuery = f"'{"', '".join(search_text.strip().split())}'"

        query = f"""
            SELECT TOP @limit
                    VectorDistance(c.vectors, @embeddings) AS SimilarityScore,
                    c.name, 
                    c.description,
                    c.categoryName,
                    c.currentPrice,
                    c.inventory,
                    c.priceHistory
            FROM c
            WHERE 
               c.docType = @docType
                AND
               VectorDistance(c.vectors, @embeddings) >= @similarity
               AND
               1=1
            ORDER BY RANK RRF(
                VectorDistance(c.vectors, @embeddings),
                FullTextScore(c.description,{fullTextQuery})
            )
        """
        parameters = [
            {"name": "@limit", "value": limit},
            {"name": "@docType", "value": "product"},
            {"name": "@embeddings", "value": embeddings},
            {"name": "@similarity", "value": similarity}
        ]

        #print(f"Full text search for: {search_text}")
        #print(f"Full text query : {fullTextQuery}")
        #print(f"Query : {query}")
        
        # Async query: gather results into a list
        products = [p async for p in ProductsContainer.query_items(
            query=query,
            parameters=parameters
        )]

        # Remove the vectors property if it appears, unnecessarily large
        for p in products:
            p.pop('vectors', None)

        return products
        
    except exceptions.CosmosHttpResponseError as e:
        logging.error(f"Cosmos DB query failed: {e}")
        raise

    except Exception as e:
        logging.error(f"Unexpected error in search_products: {e}")
        raise



### Example: hybrid vector + full‑text search

Uses `hybridsearch_products` to combine embedding similarity and full‑text score when ranking results.

In [32]:
products = await hybridsearch_products(search_text="gaming desktop with SSD storage",similarity=0.4, limit=5)
print(f"Found {len(products)} products matching the search criteria.")
display(products) # for tabular output
# pprint(products) # for json friendly output

Found 5 products matching the search criteria.


## Add a new product

### Insert a single demo product

Creates and inserts a new laptop product document into `ProductsContainer`, including metadata but no vectors yet.

In [33]:

product = {
    "id": "c912b6bc-0f7a-4f1d-b1a0-6d3f5c4d91e8",
    "docType": "product",
    "productId": "c912b6bc-0f7a-4f1d-b1a0-6d3f5c4d91e8",
    "name": "NovaTech AeroLite X15 Ultra (2025 Edition)",
    "description": "NovaTech AeroLite X15 Ultra 2025 features a 15.6-inch QHD+ IPS display, QuantumCore i9 processor, 24GB RAM, 2TB NVMe SSD, and SpectraVision RTX Graphics. Designed with an ultra-light magnesium chassis, extended battery life, WiFi 7 support, and NovaOS 14, it delivers exceptional performance for professionals and creatives on the go.",
    "categoryName": "Computers, Laptops",
    "countryOfOrigin": "Taiwan",
    "inventory": 178,
    "firstAvailable": "2025-03-10T00:00:00",
    "currentPrice": 2499.75,
    "priceHistory": [
        {
            "date": "2025-03-10T00:00:00",
            "price": 2299
        },
        {
            "date": "2025-08-15T00:00:00",
            "price": 2499.75
        }
    ]
}

try:
    created = await ProductsContainer.create_item(body=product)
    print(f"Inserted item with id: {created['id']}")

except exceptions.CosmosResourceExistsError:
    print("Item already exists, skipping insertion.")


Inserted item with id: c912b6bc-0f7a-4f1d-b1a0-6d3f5c4d91e8


## Define search text

In [34]:
search_text="Novatech X15 power 15.6 lite"

## Search with full text search alone

### Search the new product with full‑text only

Runs `fulltextsearch_products` using `search_text` to see how well pure full‑text search retrieves the newly inserted product.

In [35]:
products = await fulltextsearch_products(search_text=search_text, limit=3)
print(f"Found {len(products)} products matching the search criteria.")
display(products) # for tabular output
# pprint(products) # for json friendly output

Found 3 products matching the search criteria.


## Search with hybrid search ... opps forgot to include embeddings

### Hybrid search before adding embeddings

Runs `hybridsearch_products` while the new product has no vector, illustrating that hybrid search can miss items without embeddings.

In [36]:
products = await hybridsearch_products(search_text=search_text,similarity=0.6, limit=3)
print(f"Found {len(products)} products matching the search criteria.")
display(products) # for tabular output
# pprint(products) # for json friendly output

Found 3 products matching the search criteria.


## Add embedding and update

### Generate and attach embedding to product

Builds an embedding from the product’s name and description, adds it as `vectors`, and upserts the updated item.

In [37]:
embeddingText = f"{product['name']} {product['description']}"
    
embedding = await generate_embedding(embeddingText)
product["vectors"] = embedding

result = await ProductsContainer.upsert_item(body=product)
print(f"Upserted item with id: {result['id']}")

Upserted item with id: c912b6bc-0f7a-4f1d-b1a0-6d3f5c4d91e8


## Search with hybrid search

### Hybrid search after adding embeddings

Repeats the hybrid search to show improved ranking now that the product has a vector representation.

In [38]:
products = await hybridsearch_products(search_text=search_text,similarity=0.6, limit=3)
print(f"Found {len(products)} products matching the search criteria.")
display(products) # for tabular output
# pprint(products) # for json friendly output

Found 3 products matching the search criteria.


## Delete product

### Delete the demo product

Deletes the previously inserted product from `ProductsContainer` using its id and partition key.

In [39]:
item_id = "c912b6bc-0f7a-4f1d-b1a0-6d3f5c4d91e8"
partition_key_value = "Computers, Laptops"  
await ProductsContainer.delete_item(item=item_id, partition_key=partition_key_value)
print(f"Deleted item with id: {item_id}")

Deleted item with id: c912b6bc-0f7a-4f1d-b1a0-6d3f5c4d91e8


### Verify product deletion via search

Runs a final hybrid search with the same query to confirm the deleted product no longer appears in results.

In [40]:
products = await hybridsearch_products(search_text=search_text,similarity=0.6, limit=3)
print(f"Found {len(products)} products matching the search criteria.")
display(products) # for tabular output
# pprint(products) # for json friendly output

Found 3 products matching the search criteria.


## Bringing it back to the Raw Chunks

### Vectorized PDF raw chunks container

This block creates the `PDF_RawChunks` container for storing text chunks from PDFs along with embeddings. It:

- Uses `/id` as the partition key
- Configures a vector embedding policy for the `vectors` field (512‑dimensional `float32`, cosine distance)
- Sets an indexing policy that indexes all JSON fields but excludes the raw `vectors` array from regular indexes
- Adds a `quantizedFlat` vector index on `/vectors` for efficient vector search

The following cells load chunk data from a JSON file, generate embeddings, and upsert them into this container.

In [41]:
async def create_PDFRawChunksContainer():

    CosmosDB_ContainerName = "PDF_RawChunks"
    CosmosDB_PartitionKeyPath = "/id"
    CosmosDB_AutoscaleMax = 1000
    
    # Define the vector policy for the container
    vector_embedding_policy = {
        "vectorEmbeddings": [
            {
                "path":"/vectors",
                "dataType":"float32",
                "distanceFunction":"cosine",
                "dimensions":512
            }
        ]
    }

    # Define the indexing policy for the container
    indexing_policy = {
        "includedPaths": [
            {
                "path": "/*"
            }
        ],
        "excludedPaths": [
            {
                "path": "/vectors/*"
            },
            {
                "path": "/\"_etag\"/?"
            }
        ],

        "vectorIndexes": [
            {
                "path": "/vectors",
                "type": "quantizedFlat"
            }
        ]
    }

    # Create the vectorized sample PDF Chunks container
    container = await CosmosDBDatabase.create_container_if_not_exists(
        id=CosmosDB_ContainerName,
        partition_key=PartitionKey(path=CosmosDB_PartitionKeyPath, kind='Hash'),
        indexing_policy=indexing_policy,
        vector_embedding_policy=vector_embedding_policy,
        offer_throughput=ThroughputProperties(auto_scale_max_throughput=5000))

    print(f"Container Created")
    return container

### Create or load PDF raw chunks container

Calls `create_PDFRawChunksContainer()` and stores the resulting client in `PDFRawChunksContainer`.

In [42]:
# Run the function and get the container reference
PDFRawChunksContainer = await create_PDFRawChunksContainer()

Container Created


### Preview PDF chunk JSON file

Loads `PDF_RawChunks.JSON` from notebook resources into a pandas object and displays its contents for inspection.

In [43]:
import pandas as pd
# Load data into pandas DataFrame from f"{notebookutils.nbResPath}/builtin/PDF_RawChunks.JSON"
df = pd.read_json(f"{notebookutils.nbResPath}/builtin/PDF_RawChunks.JSON",typ="series")
display(df)

In [44]:
async def load_PDFRawChunks_from_json(file_path: str) -> List[Dict[str, Any]]:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"JSON file not found: {file_path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("Expected the JSON file to contain a top-level array of records.")

    return data


In [45]:

async def prepare_record(record: Dict[str, Any]) -> Dict[str, Any]:
    """
    Takes a record in the user's format and adds a vector field.
    """
    required_fields = ["id", "chunk_text", "source_pdf", "created_at"]
    for field in required_fields:
        if field not in record:
            raise ValueError(f"Missing required field '{field}' in record: {record}")

    chunk_text = record["chunk_text"]
    vector = await generate_embedding(chunk_text)

    enriched = dict(record)
    enriched["vectors"] = vector
    return enriched


In [46]:
async def upsert_records(records: List[Dict[str, Any]]) -> None:
    total = len(records)
    success = 0
    failures = 0

    for i, record in enumerate(records, start=1):
        try:
            item = await prepare_record(record)
            await PDFRawChunksContainer.upsert_item(item)
            success += 1
            if i % 10 == 0 or i == total:
                print(f"Processed {i}/{total} records...")
        except Exception as e:
            failures += 1
            record_id = record.get("id", "<missing id>")
            print(f"Failed record id={record_id}: {e}")

    print(f"Done. Success={success}, Failures={failures}, Total={total}")


### Load, embed, and upsert PDF chunks

Reads all records from the JSON file, generates embeddings for each chunk, and upserts them into `PDF_RawChunks`.

In [47]:

print(f"Loading JSON file: {notebookutils.nbResPath}/builtin/PDF_RawChunks.JSON")
records = await load_PDFRawChunks_from_json(f"{notebookutils.nbResPath}/builtin/PDF_RawChunks.JSON")

print(f"Loaded {len(records)} records. Generating embeddings and upserting...")
await upsert_records(records)


Loading JSON file: /synfs/nb_resource/builtin/PDF_RawChunks.JSON
Loaded 22 records. Generating embeddings and upserting...
Processed 10/22 records...
Processed 20/22 records...
Processed 22/22 records...
Done. Success=22, Failures=0, Total=22


In [48]:
#Define function to perform vector search
async def vectorsearch_PDFRawChunk(search_text: str, similarity: float = 0.7, limit: int = 5) -> List[Dict[str, Any]]:

    try:
        # Generate embeddings for the search text
        embeddings = await generate_embedding(search_text.strip())

        # Cosmos query with VectorDistance to perform similarity search
        query = """
            SELECT TOP @limit 
                VectorDistance(c.vectors, @embeddings) AS SimilarityScore,
                c.id, c.chunk_text, c.source_pdf, c.created_at
            FROM c 
            WHERE 
                VectorDistance(c.vectors, @embeddings) >= @similarity
            ORDER BY 
                VectorDistance(c.vectors, @embeddings)
        """

        parameters = [
            {"name": "@limit", "value": limit},
            {"name": "@embeddings", "value": embeddings},
            {"name": "@similarity", "value": similarity}
        ]

        # Async query: gather results into a list
        chunks = [c async for c in PDFRawChunksContainer.query_items(
            query=query,
            parameters=parameters
        )]

        # Remove the vectors property if it appears, unnecessarily large
        for c in chunks: c.pop('vectors', None)

        return chunks
        
    except exceptions.CosmosHttpResponseError as e:
        logging.error(f"Cosmos DB query failed: {e}")
        raise

    except Exception as e:
        logging.error(f"Unexpected error in search_products: {e}")
        raise


### Define a PDF question query

Stores a natural‑language question in `search_text` to search across PDF chunks.

In [49]:
search_text='What are the fees on late Payments on Credit Card';

### Run vector search over PDF chunks

Uses `vectorsearch_PDFRawChunk` to find the most relevant chunks in the PDF corpus for the given question.

In [50]:
chunks = await vectorsearch_PDFRawChunk(search_text=search_text, similarity = 0.1, limit = 5)
print(f"Found {len(chunks)} chunks matching the search criteria.")
display(chunks) # for tabular output
# pprint(chunks) # for json friendly output

Found 5 chunks matching the search criteria.
